# Member 3 — Visualization & Analysis
**Project:** circRNA–Disease Association Prediction using GraphSAGE + GNNExplainer

**Your inputs (from Member 2):**
- `results/interpretability/edge_importance.csv` — per-edge importance scores
- `results/interpretability/node_importance.csv` — per-feature importance scores
- `results/interpretability/edge_importance_bar.png` — already generated bar chart
- `results/interpretability/node_importance_bar.png` — already generated bar chart

**Your outputs (for Member 4):**
- Subgraph visualizations (NetworkX)
- Feature importance plots (Matplotlib)
- Edge type comparison plots
- Top vs Random comparison plots
- All saved to `results/member3_viz/`

## STEP 1 — Setup: Import Libraries and Load Data

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from pathlib import Path

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11

# ── Paths ─────────────────────────────────────────────────────────────────────
# Run this notebook from the project root, OR adjust ROOT below.
ROOT    = Path(".").resolve()
INTERP  = ROOT / "results" / "interpretability"   # Member 2 outputs live here
OUT_DIR = ROOT / "results" / "member3_viz"         # YOUR outputs go here
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Root          :", ROOT)
print("Reading from  :", INTERP)
print("Saving to     :", OUT_DIR)

In [ ]:
# ── Load Member 2 outputs ─────────────────────────────────────────────────────
edge_df = pd.read_csv(INTERP / "edge_importance.csv")
node_df = pd.read_csv(INTERP / "node_importance.csv")

print("edge_importance.csv shape:", edge_df.shape)
print(edge_df.head(3))
print()
print("node_importance.csv:")
print(node_df)

## STEP 2 — Plot 1: Node Feature Importance Bar Chart

**What this shows:** Which node features (degree, betweenness, node type) were most important for the model's predictions, according to GNNExplainer.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

colors = ['#7F77DD', '#9F99E8', '#AFA9EC', '#C0BBED', '#CFCBEF', '#DDDCF4']
bars = ax.barh(
    node_df['feature'][::-1],
    node_df['mean_importance'][::-1],
    xerr=node_df['std_importance'][::-1],
    color=colors,
    capsize=4,
    edgecolor='white',
    linewidth=0.5
)

# Add value labels
for bar, val in zip(bars, node_df['mean_importance'][::-1]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Mean Importance Score (GNNExplainer)')
ax.set_title('Node Feature Importance — Which features does the model rely on?')
ax.set_xlim(0, 0.55)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'plot1_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot1_feature_importance.png")

print("\nKey finding: 'degree' is the most important feature (rank 1).")
print("This means highly-connected nodes are most predictive of circRNA-disease links.")

## STEP 3 — Plot 2: Edge Type Importance Comparison

**What this shows:** Which *types* of connections (circRNA→disease, miRNA→disease, etc.) carry the most importance. This tells us which biological relationships the model learned to use.

In [ ]:
# Average importance by edge type pair
edge_type_df = (
    edge_df
    .groupby(['edge_src_type', 'edge_dst_type'])['importance']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .sort_values('mean', ascending=False)
)
edge_type_df['edge_type'] = (
    edge_type_df['edge_src_type'] + ' → ' + edge_type_df['edge_dst_type']
)

# Color map: one color per node type
color_map = {
    'circRNA': '#7F77DD',
    'miRNA':   '#1D9E75',
    'disease': '#D85A30'
}
bar_colors = [color_map[r] for r in edge_type_df['edge_src_type']]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(
    edge_type_df['edge_type'],
    edge_type_df['mean'],
    yerr=edge_type_df['std'],
    color=bar_colors,
    capsize=4,
    edgecolor='white',
    linewidth=0.5
)

ax.set_ylabel('Mean Edge Importance')
ax.set_title('Importance by Edge Type — Which biological connections matter most?')
ax.set_xticklabels(edge_type_df['edge_type'], rotation=30, ha='right')

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in color_map.items()]
ax.legend(handles=legend_patches, title='Source node type')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'plot2_edge_type_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot2_edge_type_importance.png")

## STEP 4 — Plot 3: Top vs Random Predictions Comparison

**What this shows:** Does the model assign higher importance to edges in *high-confidence* predictions vs *random* ones? If yes, GNNExplainer is working correctly.

In [ ]:
top_imp    = edge_df[edge_df['group'] == 'top']['importance']
random_imp = edge_df[edge_df['group'] == 'random']['importance']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: histogram comparison ────────────────────────────────────────────────
axes[0].hist(top_imp,    bins=40, alpha=0.7, color='#7F77DD', label='Top predictions')
axes[0].hist(random_imp, bins=40, alpha=0.7, color='#1D9E75', label='Random predictions')
axes[0].set_xlabel('Edge Importance Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Edge Importance\n(Top vs Random predictions)')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Right: box plot comparison ────────────────────────────────────────────────
data   = [top_imp.values, random_imp.values]
labels = ['Top\npredictions', 'Random\npredictions']
bp = axes[1].boxplot(data, labels=labels, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#7F77DD', '#1D9E75']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Edge Importance Score')
axes[1].set_title('Edge Importance Spread\n(Top vs Random predictions)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUT_DIR / 'plot3_top_vs_random.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot3_top_vs_random.png")
print(f"Top mean: {top_imp.mean():.4f} | Random mean: {random_imp.mean():.4f}")

## STEP 5 — Plot 4: Top 20 Most Important Specific Edges

**What this shows:** The actual named biological connections (e.g., specific circRNA → disease) that the model finds most important. This is the key result for Member 4 to interpret biologically.

In [ ]:
# Aggregate: average importance per unique (src_name, dst_name) pair
top_edges = (
    edge_df
    .groupby(['edge_src_name', 'edge_dst_name', 'edge_src_type', 'edge_dst_type'])['importance']
    .mean()
    .reset_index()
    .sort_values('importance', ascending=False)
    .head(20)
)

top_edges['label'] = top_edges['edge_src_name'] + '  →  ' + top_edges['edge_dst_name']

color_map = {'circRNA': '#7F77DD', 'miRNA': '#1D9E75', 'disease': '#D85A30'}
bar_colors = [color_map[t] for t in top_edges['edge_src_type']]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_edges['label'][::-1], top_edges['importance'][::-1],
               color=bar_colors[::-1], edgecolor='white', linewidth=0.5)

ax.set_xlabel('Mean Edge Importance Score')
ax.set_title('Top 20 Most Important Biological Connections\n(as identified by GNNExplainer)')

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in color_map.items()]
ax.legend(handles=legend_patches, title='Source type', loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'plot4_top20_edges.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot4_top20_edges.png")

## STEP 6 — Plot 5: Subgraph Visualization using NetworkX

**What this shows:** A visual graph of the most important biological connections for a single high-confidence prediction. Nodes are colored by type (circRNA / miRNA / disease). Edge thickness shows importance.

In [ ]:
def build_subgraph(pair_idx, edge_df, top_n_edges=30):
    """Build a NetworkX subgraph for one explained prediction."""
    pair_edges = edge_df[edge_df['pair_idx'] == pair_idx].copy()
    
    # Keep only the top N edges by importance to keep graph readable
    pair_edges = pair_edges.nlargest(top_n_edges, 'importance')

    G = nx.DiGraph()
    for _, row in pair_edges.iterrows():
        G.add_node(row['edge_src_name'], node_type=row['edge_src_type'])
        G.add_node(row['edge_dst_name'], node_type=row['edge_dst_type'])
        G.add_edge(
            row['edge_src_name'], row['edge_dst_name'],
            weight=row['importance']
        )
    return G


def plot_subgraph(G, pair_idx, title_extra='', save_path=None):
    """Draw a subgraph with color-coded nodes and weighted edges."""
    color_map = {'circRNA': '#7F77DD', 'miRNA': '#1D9E75', 'disease': '#D85A30'}
    default_color = '#AAAAAA'

    node_colors = [
        color_map.get(G.nodes[n].get('node_type', ''), default_color)
        for n in G.nodes()
    ]
    edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
    # Normalize edge width 1–5
    w_min, w_max = min(edge_weights), max(edge_weights)
    def norm_w(w):
        if w_max == w_min:
            return 2.0
        return 1.0 + 4.0 * (w - w_min) / (w_max - w_min)
    edge_widths = [norm_w(w) for w in edge_weights]

    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(G, seed=42, k=1.5)

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=600,
                           ax=ax, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(
        G, pos,
        width=edge_widths,
        alpha=0.6,
        edge_color='#555555',
        arrows=True,
        arrowsize=15,
        ax=ax
    )

    legend_patches = [mpatches.Patch(color=c, label=t) for t, c in color_map.items()]
    ax.legend(handles=legend_patches, loc='upper left', fontsize=9)
    ax.set_title(f'Subgraph for prediction pair_idx={pair_idx}\n'
                 f'{title_extra}Edge thickness = GNNExplainer importance',
                 fontsize=11)
    ax.axis('off')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {Path(save_path).name}")
    plt.show()


# ── Plot for the TOP prediction (pair_idx=119 is the highest-confidence pair) ─
top_pair_idx = 119   # change this to any pair_idx from edge_df['pair_idx'].unique()
G_top = build_subgraph(top_pair_idx, edge_df, top_n_edges=25)
print(f"Subgraph nodes: {G_top.number_of_nodes()} | edges: {G_top.number_of_edges()}")
plot_subgraph(G_top, top_pair_idx,
              save_path=OUT_DIR / 'plot5_subgraph_top_prediction.png')

## STEP 7 — Plot 6: Multiple Subgraphs (Top 4 Predictions)

**What this shows:** Side-by-side subgraphs for the top 4 high-confidence predictions, so Member 4 can compare which connections appear repeatedly.

In [ ]:
top_pairs = sorted(edge_df[edge_df['group'] == 'top']['pair_idx'].unique())[:4]
print("Visualizing pair_idx:", top_pairs)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

color_map = {'circRNA': '#7F77DD', 'miRNA': '#1D9E75', 'disease': '#D85A30'}

for ax, pair_idx in zip(axes, top_pairs):
    G = build_subgraph(pair_idx, edge_df, top_n_edges=20)
    
    node_colors = [
        color_map.get(G.nodes[n].get('node_type', ''), '#AAAAAA')
        for n in G.nodes()
    ]
    edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
    w_min, w_max = min(edge_weights), max(edge_weights)
    edge_widths = [
        1.0 + 4.0 * (w - w_min) / (w_max - w_min + 1e-9)
        for w in edge_weights
    ]

    pos = nx.spring_layout(G, seed=42, k=1.2)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=400,
                           ax=ax, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=6, ax=ax)
    nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.6,
                           edge_color='#555555', arrows=True,
                           arrowsize=12, ax=ax)
    ax.set_title(f'pair_idx = {pair_idx}', fontsize=10)
    ax.axis('off')

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in color_map.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           title='Node type')
fig.suptitle('Explanation Subgraphs — Top 4 High-Confidence Predictions\n'
             'Edge thickness = GNNExplainer importance score', fontsize=13)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig(OUT_DIR / 'plot6_top4_subgraphs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot6_top4_subgraphs.png")

## STEP 8 — Plot 7: Important vs Non-Important Connections Comparison

**What this shows:** Directly comparing high-importance edges vs low-importance edges by edge type. This answers: "What kinds of connections are the model actually using vs ignoring?"

In [ ]:
# Split edges into High importance (top 25%) and Low importance (bottom 25%)
threshold_high = edge_df['importance'].quantile(0.75)
threshold_low  = edge_df['importance'].quantile(0.25)

high_imp = edge_df[edge_df['importance'] >= threshold_high].copy()
low_imp  = edge_df[edge_df['importance'] <= threshold_low].copy()

high_imp['edge_type'] = high_imp['edge_src_type'] + ' → ' + high_imp['edge_dst_type']
low_imp['edge_type']  = low_imp['edge_src_type']  + ' → ' + low_imp['edge_dst_type']

high_counts = high_imp['edge_type'].value_counts(normalize=True) * 100
low_counts  = low_imp['edge_type'].value_counts(normalize=True) * 100

# Align categories
all_types   = sorted(set(high_counts.index) | set(low_counts.index))
high_vals   = [high_counts.get(t, 0) for t in all_types]
low_vals    = [low_counts.get(t, 0)  for t in all_types]

x = np.arange(len(all_types))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, high_vals, width, label=f'High importance (top 25%)',
       color='#7F77DD', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, low_vals,  width, label=f'Low importance (bottom 25%)',
       color='#AAAAAA', alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(all_types, rotation=30, ha='right')
ax.set_ylabel('Percentage of edges (%)')
ax.set_title('Important vs Non-Important Connections by Edge Type\n'
             '(What types of connections does the model rely on?)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'plot7_important_vs_nonimportant.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot7_important_vs_nonimportant.png")

## STEP 9 — Plot 8: Feature Importance Radar Chart

**What this shows:** A radar/spider chart summarizing all 6 node features at once. Easier to read at a glance than a bar chart.

In [ ]:
features = node_df['feature'].tolist()
values   = node_df['mean_importance'].tolist()

# Radar chart requires closing the loop
N      = len(features)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]       # close the polygon
values_plot = values + values[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

ax.plot(angles, values_plot, 'o-', linewidth=2, color='#7F77DD')
ax.fill(angles, values_plot, alpha=0.25, color='#7F77DD')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(features, fontsize=10)
ax.set_ylim(0, 0.55)
ax.set_title('Node Feature Importance — Radar View', pad=20, fontsize=12)
ax.grid(color='#DDDDDD', linewidth=0.5)

plt.tight_layout()
plt.savefig(OUT_DIR / 'plot8_feature_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot8_feature_radar.png")

## STEP 10 — Summary: Print All Output Files

After running all cells, check this summary — then hand these files to Member 4.

In [ ]:
print("="*60)
print("MEMBER 3 — ALL OUTPUT FILES")
print("="*60)
all_files = sorted(OUT_DIR.glob('*.png'))
for f in all_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s}  {size_kb:6.1f} KB")

print()
print("KEY FINDINGS SUMMARY (for Member 4):")
print("-"*60)
print(f"  Top feature: {node_df.iloc[0]['feature']} "
      f"(importance = {node_df.iloc[0]['mean_importance']:.3f})")
print(f"  Total unique explained pairs: {edge_df['pair_idx'].nunique()}")
print(f"  Total edges analyzed: {len(edge_df):,}")

top_edge = (
    edge_df.groupby(['edge_src_name','edge_dst_name'])['importance']
    .mean().idxmax()
)
print(f"  Most important specific edge: {top_edge[0]}  →  {top_edge[1]}")
print()
print("Pass the results/member3_viz/ folder to Member 4.")
print("="*60)